In [5]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & artiq_run " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, freq_raman, img_amp):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                # self.p.v_pd_hf_tweezer_1064_rampdown3_end = 3.8
                # self.t_hf_tweezer_1064_rampdown3 = 10.e-3
                
                # self.xvar('beans',[0,1]*50)

                self.p.i_hf_raman = 182.

                self.p.v_pd_hf_tweezer_squeeze_power = 6.

                self.p.t_tweezer_squeezer_ramp_1 = 10.e-3
                self.p.t_tweezer_squeezer_ramp_2 = 90.e-3

                # self.xvar('frequency_raman_transition',147.311e6 + np.linspace(-150.e3,150.e3,15))

                self.p.frequency_raman_transition = {freq_raman}

                # self.xvar('amp_raman',np.linspace(0.1,.35,15))
                self.p.fraction_power_raman = .5

                # self.xvar('t_raman_stateprep_pulse',[0.,9.9979e-06])
                self.p.t_raman_stateprep_pulse = (1.0058e-05) / 2

                # self.xvar('t_continuous_rabi',np.linspace(0.,400.e-6,10))
                self.p.t_continuous_rabi = 300.e-6

                # self.xvar('t_raman_pulse',[0.,5.2774e-06 / 2, 5.2774e-06])
                self.p.t_raman_pulse = 5.2774e-06
                
                # self.xvar('amp_imaging',np.linspace(0.5,1.9,30))
                self.p.amp_imaging = {img_amp}

                self.p.hf_imaging_detuning = -568.e6 # 182.

                # self.xvar('hf_imaging_detuning_mid',np.arange(-680.,-610.,5)*1.e6)
                self.p.hf_imaging_detuning_mid = -630.e6 # -635.e6
                
                # self.xvar('dimension_slm_mask',np.linspace(15.e-6,250.e-6,10))
                self.p.dimension_slm_mask = 20.e-6

                # self.xvar('phase_slm_mask',np.linspace(0.*np.pi,1.3*np.pi,15))
                self.p.phase_slm_mask = 0.42 * np.pi

                # self.xvar('t_tweezer_hold',np.linspace(1.e-3,1.1e-3,10))
                self.p.t_tweezer_hold = 20.e-3
                self.p.t_tof = 20.e-6
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 10

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning_mid)
                # self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)

                # self.raman.pulse(t=self.p.t_raman_pulse)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(5.e-6)
                self.raman.pulse(t=self.p.t_continuous_rabi)
                # delay(50.e-6)
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [6]:
eBuilder = ExptBuilder()

In [7]:
img_amp = np.linspace(.5,1.9,30)
freq_raman = np.array([1.47274133e+08, 1.47275527e+08, 1.47276920e+08, 1.47278314e+08,
       1.47279708e+08, 1.47281101e+08, 1.47282495e+08, 1.47283888e+08,
       1.47285282e+08, 1.47286675e+08, 1.47288069e+08, 1.47289463e+08,
       1.47290856e+08, 1.47292250e+08, 1.47293643e+08, 1.47295037e+08,
       1.47296430e+08, 1.47297824e+08, 1.47299217e+08, 1.47300611e+08,
       1.47302005e+08, 1.47303398e+08, 1.47304792e+08, 1.47306185e+08,
       1.47307579e+08, 1.47308972e+08, 1.47310366e+08, 1.47311760e+08,
       1.47313153e+08, 1.47314547e+08])
# np.random.shuffle(ts)
for f in range(30):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman=freq_raman[f]))
    eBuilder.run_expt()

0.5 147274133.0
0 Run id: 62261
 10 values of dummy. 10 total shots. 30 total images expected.
B:\_K\PotassiumData\2026-03-29\0062261_2026-03-29_10-56-09_hf_monitored_rabi.hdf5
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.42, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.42 pi, x-center = 993, y-center = 820

 Run ID: 62261

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.42, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.42 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.42, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.42 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.42, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}


In [13]:

relay = ethernet_relay

In [14]:
relay.source_off()

AttributeError: module 'waxx.control.ethernet_relay' has no attribute 'source_off'